Okay also hier möchte ich vorallem mit Sympy sachen diagonalisieren und gleichzeitig easy auslesen können in markdown. Mal sehen obs klappt...

In [47]:
import sympy
from sympy import *
from IPython.display import display, Math, Markdown
import math

# create symbolic variables
f= sympy.symbols('f')
m = symbols("m", real=True)
phi = symbols('phi', real=True)
Ep, Em, lp, lm = sympy.symbols('E_+, E_-, lambda_+, lambda_-', real=True)
f_conj = sympy.conjugate(f)
Delta = sympy.symbols('Delta', real=True)


# Keep readable names as symbols, but also store explicit definitions
# Ep = sympy.Symbol('Ep')
# Em = sympy.Symbol('Em')
# lp = sympy.Symbol('lp')
# lm = sympy.Symbol('lm')

lp_def = m + sqrt(f*f_conj)
lm_def = m - sqrt(f*f_conj)
Ep_def = sqrt(Delta**2 + lp_def**2)
Em_def = sqrt(Delta**2 + lm_def**2)



# Use this mapping when you want SymPy to expand them
defs = {Ep: Ep_def, Em: Em_def, lp: lp_def, lm: lm_def}
# matrix: [[A, -f], [-conjugate(f), B]]


# Delta = 0
# M = sympy.Matrix([[m, -f_conj, Delta, 0], [-f, m, 0 ,Delta], [Delta, 0, -m, f_conj],[0, Delta, f, -m]])

# # diagonalize: M = P * D * P.inv()
# P, D = M.diagonalize()

# display(Math(f"D = {sympy.latex(D)}"))
# display(Math(f"P = {sympy.latex(sympy.simplify(P))}"))
# display(Math(sympy.latex(sympy.simplify(P*D*P**(-1)))))
# print('Matrix M:')
# print(M)
# print('\nP (eigenvectors):')
# print(P)
# print('\nD (diagonal matrix of eigenvalues):')
# print(D)



""" Jetzt einmal mit dem was ich raushabe alles multiplizieren und gucken was raus kommt."""
W = sympy.Matrix([
    [-sympy.exp(-sympy.I*phi), sympy.exp(-sympy.I*phi), 0, 0],
    [1, 1, 0, 0],
    [0, 0, -sympy.exp(-sympy.I*phi), sympy.exp(-sympy.I*phi)],
    [0, 0, 1, 1],
]) / sympy.sqrt(2)
P = sympy.Matrix([[1,0,0,0],
                [0,0,1,0],
                [0,1,0,0],
                [0,0,0,1]])
W_2 = sympy.Matrix([[Delta/sqrt(Ep**2 + lp*Ep),Delta/sqrt(Ep**2 - lp*Ep),0,0],[(Ep-lp)/sqrt(Ep**2 + lp*Ep),-(Ep+lp)/sqrt(Ep**2 - lp*Ep),0,0],[0,0,Delta/sqrt(Em**2 + lm*Em),Delta/sqrt(Em**2 - lm*Em)],[0,0,(Em-lm)/sqrt(Em**2 + lm*Em),-(Em+lm)/sqrt(Em**2 - lm*Em)]]) / sqrt(2)

# Python evaluates * left-to-right; parenthesize to force grouping (though matrix multiplication is associative)
U = simplify(W*P*W_2)

print(latex(U))

# display(Math(f"W= {sympy.latex(sympy.simplify(W))}"))
# display(Math(f"P = {sympy.latex(P)}"))
# display(Math(f"W\' = {sympy.latex(sympy.simplify(W_2))}"))
display(Math(f"U = {sympy.latex(sympy.simplify(U))}"))


# BdG-Matrix für Delta != 0
M = Matrix([
    [m,      -f_conj,  Delta,   0],
    [-f,      m,       0,      Delta],
    [Delta,   0,      -m,       f_conj],
    [0,      Delta,    f,      -m],
])

# U^\dagger M U
UdMU = (U.H * M * U).applyfunc(simplify)
display(Math(r"U^\dagger M U = " + latex(UdMU)))

# Mit expliziten Definitionen + Realitätsannahmen für Konjugierte
r = symbols('r', real=True, nonnegative=True)

real_conj_subs = {
    conjugate(Ep): Ep,
    conjugate(Em): Em,
    conjugate(lp): lp,
    conjugate(lm): lm,
    conjugate(r): r,
    conjugate(1/Ep): 1/Ep,
    conjugate(1/Em): 1/Em,
    conjugate(1/lp): 1/lp,
    conjugate(1/lm): 1/lm
}

radial_subs = {
    sqrt(f * f_conj): r,
    sqrt(f * conjugate(f)): r,
    conjugate(sqrt(f * f_conj)): r,
    conjugate(sqrt(f * conjugate(f))): r,
    f : r*exp(I *phi),
    f_conj: r*exp(-I*phi)
}

UdMU_explicit = simplify(
    UdMU
    .subs(real_conj_subs)
    .subs(defs)
    .subs(radial_subs)
)

# display(Math(r"U^\dagger M U\;\text{(mit Definitionen, } \sqrt{f\bar f}\to r\text{)} = " + latex(UdMU_explicit)))
print(latex(UdMU_explicit))

# enforce reality of the various roots/reciprocals and simplify to the claimed diagonal form
s1 = Delta**2 + (m + r)**2
s2 = Delta**2 + (m - r)**2

# lp_def = m + sqrt(f*f_conj)
# lm_def = m - sqrt(f*f_conj)
# Ep_def = sqrt(Delta**2 + lp_def**2)
# Em_def = sqrt(Delta**2 + lm_def**2)
Sp = sqrt(Delta**2 + (m+r)**2)

conj_map = {
    sympy.conjugate(s1**(-sympy.Rational(1,4))): s1**(-sympy.Rational(1,4)),
    sympy.conjugate(s2**(-sympy.Rational(1,4))): s2**(-sympy.Rational(1,4)),
    sympy.conjugate(1/sympy.sqrt(m + r + sympy.sqrt(s1))): 1/sympy.sqrt(m + r + sympy.sqrt(s1)),
    sympy.conjugate(1/sympy.sqrt(-m - r + sympy.sqrt(s1))): 1/sympy.sqrt(-m - r + sympy.sqrt(s1)),
    sympy.conjugate(1/sympy.sqrt(m - r + sympy.sqrt(s2))): 1/sympy.sqrt(m - r + sympy.sqrt(s2)),
    sympy.conjugate(1/sympy.sqrt(-m + r + sympy.sqrt(s2))): 1/sympy.sqrt(-m + r + sympy.sqrt(s2)),
    m + r : lp,
    m - r : lm,
    sqrt(Delta**2 + (m+r)**2): Ep,
    sqrt(Delta**2 + (m - r)**2): Em,
    # Delta: sqrt(Sp-(m+r))*(Sp+(m+r))
}

identity_map = {
    sqrt(Delta**2 + lp**2): Ep,
    sqrt(Delta**2 + lm**2): Em,
    Delta**2: Ep**2 - lp**2
}

# UdMU_real = UdMU_explicit.expand()
UdMU_real = simplify(UdMU_explicit.subs(real_conj_subs).subs(conj_map))
UdMU_real = simplify(UdMU_real.subs(identity_map))

target = sympy.diag(Ep, -Ep, Em, -Em)
diff = simplify(UdMU_real - target)

display(Math(r"U^\dagger M U\ (\text{real}) = " + latex(UdMU_real)))
print(latex(UdMU_real))
display(Math(r"U^\dagger M U - \mathrm{diag}(E_+,-E_+,E_-,-E_-) = " + latex(diff)))


# Optional: Test auf Diagonalform
# offdiag = (UdMU_explicit - diag(*UdMU_explicit.diagonal())).applyfunc(simplify)
# display(Math(r"\text{Offdiag}(U^\dagger M U) = " + latex(offdiag)))

\left[\begin{matrix}- \frac{\Delta e^{- i \phi}}{2 \sqrt{E_{+} \left(E_{+} + \lambda_{+}\right)}} & - \frac{\Delta e^{- i \phi}}{2 \sqrt{E_{+} \left(E_{+} - \lambda_{+}\right)}} & \frac{\Delta e^{- i \phi}}{2 \sqrt{E_{-} \left(E_{-} + \lambda_{-}\right)}} & \frac{\Delta e^{- i \phi}}{2 \sqrt{E_{-} \left(E_{-} - \lambda_{-}\right)}}\\\frac{\Delta}{2 \sqrt{E_{+} \left(E_{+} + \lambda_{+}\right)}} & \frac{\Delta}{2 \sqrt{E_{+} \left(E_{+} - \lambda_{+}\right)}} & \frac{\Delta}{2 \sqrt{E_{-} \left(E_{-} + \lambda_{-}\right)}} & \frac{\Delta}{2 \sqrt{E_{-} \left(E_{-} - \lambda_{-}\right)}}\\\frac{\left(- E_{+} + \lambda_{+}\right) e^{- i \phi}}{2 \sqrt{E_{+} \left(E_{+} + \lambda_{+}\right)}} & \frac{\left(E_{+} + \lambda_{+}\right) e^{- i \phi}}{2 \sqrt{E_{+} \left(E_{+} - \lambda_{+}\right)}} & \frac{\left(E_{-} - \lambda_{-}\right) e^{- i \phi}}{2 \sqrt{E_{-} \left(E_{-} + \lambda_{-}\right)}} & \frac{\left(- E_{-} - \lambda_{-}\right) e^{- i \phi}}{2 \sqrt{E_{-} \left(E_{-} - \lambda_{

<IPython.core.display.Math object>

<IPython.core.display.Math object>

\left[\begin{matrix}\frac{\left(\Delta^{2} \sqrt{\Delta^{2} + \left(m + r\right)^{2}} - \left(\Delta^{2} + m \left(m + r - \sqrt{\Delta^{2} + \left(m + r\right)^{2}}\right) + r \left(m + r - \sqrt{\Delta^{2} + \left(m + r\right)^{2}}\right)\right) \left(m + r - \sqrt{\Delta^{2} + \left(m + r\right)^{2}}\right)\right) \overline{\frac{1}{\sqrt[4]{\Delta^{2} + \left(m + r\right)^{2}}}} \overline{\frac{1}{\sqrt{m + r + \sqrt{\Delta^{2} + \left(m + r\right)^{2}}}}}}{2 \sqrt[4]{\Delta^{2} + \left(m + r\right)^{2}} \sqrt{m + r + \sqrt{\Delta^{2} + \left(m + r\right)^{2}}}} & 0 & 0 & 0\\0 & - \frac{\left(\Delta^{2} \sqrt{\Delta^{2} + \left(m + r\right)^{2}} + \left(\Delta^{2} + m \left(m + r + \sqrt{\Delta^{2} + \left(m + r\right)^{2}}\right) + r \left(m + r + \sqrt{\Delta^{2} + \left(m + r\right)^{2}}\right)\right) \left(m + r + \sqrt{\Delta^{2} + \left(m + r\right)^{2}}\right)\right) \overline{\frac{1}{\sqrt[4]{\Delta^{2} + \left(m + r\right)^{2}}}} \overline{\frac{1}{\sqrt{- m - r + \sqrt{\

<IPython.core.display.Math object>

\left[\begin{matrix}\frac{E_{+} \left(E_{+}^{2} - \lambda_{+}^{2}\right) - \left(E_{+} - \lambda_{+}\right) \left(- E_{+}^{2} + \lambda_{+}^{2} + m \left(E_{+} - \lambda_{+}\right) + r \left(E_{+} - \lambda_{+}\right)\right)}{2 E_{+} \left(E_{+} + \lambda_{+}\right)} & 0 & 0 & 0\\0 & \frac{- E_{+} \left(E_{+}^{2} - \lambda_{+}^{2}\right) - \left(E_{+} + \lambda_{+}\right) \left(E_{+}^{2} - \lambda_{+}^{2} + m \left(E_{+} + \lambda_{+}\right) + r \left(E_{+} + \lambda_{+}\right)\right)}{2 E_{+} \left(E_{+} - \lambda_{+}\right)} & 0 & 0\\0 & 0 & \frac{E_{-} \left(E_{+}^{2} - \lambda_{+}^{2}\right) + \left(E_{-} - \lambda_{-}\right) \left(E_{+}^{2} - \lambda_{+}^{2} - m \left(E_{-} - \lambda_{-}\right) + r \left(E_{-} - \lambda_{-}\right)\right)}{2 E_{-} \left(E_{-} + \lambda_{-}\right)} & 0\\0 & 0 & 0 & \frac{- E_{-} \left(E_{+}^{2} - \lambda_{+}^{2}\right) - \left(E_{-} + \lambda_{-}\right) \left(E_{+}^{2} - \lambda_{+}^{2} + m \left(E_{-} + \lambda_{-}\right) - r \left(E_{-} + \lambda_

<IPython.core.display.Math object>